In [8]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.append("C:/Users/BeaBa/Documents/GitHub/si/src")
sys.path.append("C:/Users/BeaBa/Documents/GitHub/si/src/metrics")
from si.statistics.euclidean_distance import euclidean_distance
from metrics.rmse import RMSE



ModuleNotFoundError: No module named 'metrics'

In [38]:
from typing import Callable, Union

import numpy as np

from si.base.model import Model
from si.data.dataset import Dataset
from si.metrics.rmse import rmse
from si.statistics.euclidean_distance import euclidean_distance


class KNNRegressor(Model):
    """
    KNN regression is a non-parametric machine learning method suitable for regression problems.
    This method classifies new sample based on a similarity measure, predicting the class of
    the new sample by looking at the values of the k-nearest samples in the training data.
    """

    def __init__(self, k: int = 1, distance: Callable = euclidean_distance, **kwargs):
        """
        Initialize the KNN classifier

        Parameters
        ----------
        k: int
            The number of k nearest example to consider
        distance: Callable
            Function that calculates the distance between a sample and the samples
            in the training dataset
        """
        super().__init__(**kwargs)
        self.k = k
        self.distance = distance

        self.dataset = None

    def _fit(self, dataset: Dataset) -> 'KNNRegressor':
        """
        Fits the model to the given dataset

        Parameters
        ----------
        dataset: Dataset
            The dataset to fit the model to (training dataset)

        Returns
        -------
        self: KNNRegressor
            The fitted model
        """
        self.dataset = dataset
        return self

    def _get_closest_value(self, sample: np.ndarray) -> Union[int, float]:
        """
        It returns the closest label of the given sample

        Parameters
        ----------
        sample: np.ndarray
            The sample to get the closest value of

        Returns
        -------
        value: int or float
            The closest value
        """
        # Get the training data
        X_train = self.dataset.X
        y_train = self.dataset.y

        # Compute the distance between the sample and the training dataset
        distances = self.distance(sample, X_train)

        # Get the k nearest neighbors
        k_nearest_neighbors = np.argsort(distances)[:self.k]

        # Get the values of the k nearest neighbors
        k_nearest_neighbors_label_values = y_train[k_nearest_neighbors]

        # Get the average value of the k nearest neighbors
        value = np.sum(k_nearest_neighbors_label_values) / self.k

        return value

    def _predict(self, dataset: Dataset) -> np.ndarray:
        """
        Predict values for the test dataset.

        Parameters
        ----------
        dataset: Dataset
            The dataset to predict the values of (testing dataset)

        Returns
        -------
        predictions: np.ndarray
            An array of predicted values for the testing dataset
        """
        
        # Get the training data
        X_train = self.dataset.X
        y_train = self.dataset.y

        # Initialize predictions array
        predictions = []

        # Iterate over each test sample and get the predicted value
        for sample in dataset.X:
            prediction = self._get_closest_value(sample)
            predictions.append(prediction)

        return np.array(predictions)

    def _score(self, dataset: Dataset) -> float:
        """
        Computes the root mean squared error between the estimated values and the true values of a given dataset

        Parameters
        ----------
        dataset: Dataset
            The dataset to evaluate the model on

        Returns
        -------
        float
            The root mean squared error of the model for the given dataset
        """
        predictions = self._predict(dataset)
        return rmse(dataset.y, predictions)


In [15]:
from si.io.csv_file import read_csv
import pandas as pd

Path= "C:/Users/BeaBa/Documents/GitHub/si/datasets/cpu/"
cpu = pd.read_csv(Path + "cpu.csv", sep=",", index_col=False)

In [16]:
print (cpu.head())

   syct  mmin   mmax  cach  chmin  chmax  perf
0   125   256   6000   256     16    128   198
1    29  8000  32000    32      8     32   269
2    29  8000  32000    32      8     32   220
3    29  8000  32000    32      8     32   172
4    29  8000  16000    32      8     16   132


In [39]:
from sklearn.model_selection import train_test_split

X = cpu.to_numpy()[:, :-1]  
y = cpu.to_numpy()[:, -1]   


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [40]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [42]:
# Assume Dataset is imported and works as expected
from si.data.dataset import Dataset

# Prepare dataset objects
train_dataset = Dataset(X_train, y_train)
test_dataset = Dataset(X_test, y_test)

# Instantiate and train the model
knn = KNNRegressor(k=2, distance=euclidean_distance)
knn._fit(train_dataset)

# Make predictions
predictions = knn._predict(test_dataset)

# Evaluate the model
score = knn._score(test_dataset)

# Print the results
print("Previções:", predictions)
print("RMSE:", score)


Previções: [177.5  17.5  33.  165.   17.  323.5  66.   26.  573.  171.5  89.  123.
  22.5  46.   41.5  33.   89.5  29.  134.   73.   48.5  94.  108.5  16.5
  14.5  40.   21.5  25.5  36.   61.  323.5  28.5  27.5  19.   25.5  39.
  42.5 428.   34.   20.5 401.   33.  152.5 428.  388.5  22.  325.   25.5
  23.   40.   32.5  25.   45.   26.  178.   89.   25.   10.  144.  323.5
 108.5  49.  196.   73.   57.5  39.   28.5 177.5  61.  160.   33.5  63.
 102.  152.5  66.   19.   61.   40.   22.   63.  106.5  33.   19.   32.5]
RMSE: 138.9599119993142
